# Twitter Sentiment Analysis — Full Project

This **Google Colab** notebook reproduces the beginner project: 10,000 NLTK tweets → cleaning → tokenization → split → TF-IDF → Naive Bayes → evaluation → custom predictions → saved model → optional Gradio interface.

**Labels:** `1 = Positive`, `0 = Negative`. The project is binary, not a neutral/three-way sentiment classifier.

> The first run requires internet access to download NLTK `twitter_samples`. Run all cells in order.

## 1. Install requirements

In [ ]:
%pip -q install nltk scikit-learn matplotlib cloudpickle gradio

## 2. Download Twitter samples and inspect their sizes

In [ ]:
import nltk
nltk.download('twitter_samples')
from nltk.corpus import twitter_samples

positive_tweets = twitter_samples.strings('positive_tweets.json')
negative_tweets = twitter_samples.strings('negative_tweets.json')

print('Positive tweets:', len(positive_tweets))
print('Negative tweets:', len(negative_tweets))
print('Total tweets:', len(positive_tweets) + len(negative_tweets))

## 3. Explore the dataset

In [ ]:
print('=== POSITIVE TWEETS ===')
for tweet in positive_tweets[:3]:
    print(tweet)
    print('-' * 30)
print('\n=== NEGATIVE TWEETS ===')
for tweet in negative_tweets[:3]:
    print(tweet)
    print('-' * 30)

## 4. Clean tweets

Remove URLs and @mentions, lowercase text, preserve hashtag words and sentiment-bearing emoticons.

In [ ]:
import re

def clean_tweet(tweet):
    tweet = tweet.lower()
    tweet = re.sub(r'https?://\S+|www\.\S+', '', tweet)
    tweet = re.sub(r'@\w+', '', tweet)
    tweet = re.sub(r'#(\w+)', r'\1', tweet)
    tweet = re.sub(r'\s+', ' ', tweet).strip()
    return tweet

clean_positive = [clean_tweet(t) for t in positive_tweets]
clean_negative = [clean_tweet(t) for t in negative_tweets]
print('Original:', positive_tweets[0])
print('Cleaned: ', clean_positive[0])
print('Negative:', clean_negative[0])

## 5. Tokenize tweets

In [ ]:
from nltk.tokenize import TweetTokenizer

tokenizer = TweetTokenizer()
tokenized_positive = [tokenizer.tokenize(t) for t in clean_positive]
tokenized_negative = [tokenizer.tokenize(t) for t in clean_negative]
print('Positive tokens:', tokenized_positive[0])
print('Negative tokens:', tokenized_negative[0])

## 6. Split into train and test sets (80% / 20%)

In [ ]:
from sklearn.model_selection import train_test_split

X = tokenized_positive + tokenized_negative
y = [1] * len(tokenized_positive) + [0] * len(tokenized_negative)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('Training tweets:', len(X_train))
print('Testing tweets:', len(X_test))
print('Training labels:', {0: y_train.count(0), 1: y_train.count(1)})
print('Testing labels:', {0: y_test.count(0), 1: y_test.count(1)})

## 7. TF-IDF features

Our input is already tokenized. Learn the vocabulary **only on the training set** to prevent leakage.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    analyzer=lambda tweet: tweet,
    lowercase=False
)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)
print('Training matrix:', X_train_tfidf.shape)
print('Testing matrix: ', X_test_tfidf.shape)

## 8. Train the Multinomial Naive Bayes classifier

In [ ]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()
model.fit(X_train_tfidf, y_train)
print('Model trained! Classes:', model.classes_)

## 9. Evaluate test performance

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy * 100:.2f}%')
print(classification_report(
    y_test, y_pred, target_names=['Negative', 'Positive']
))

## 10. Plot a confusion matrix

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, y_pred)
print('Confusion matrix:\n', cm)
disp = ConfusionMatrixDisplay(cm, display_labels=['Negative', 'Positive'])
disp.plot(cmap='Blues', values_format='d')
plt.title('Twitter Sentiment Analysis — Confusion Matrix')
plt.show()

## 11. Examine incorrect predictions

In [ ]:
incorrect_indices = [i for i in range(len(y_test)) if y_test[i] != y_pred[i]]
print('Incorrect predictions:', len(incorrect_indices))
for i in incorrect_indices[:5]:
    actual = 'Positive' if y_test[i] == 1 else 'Negative'
    pred = 'Positive' if y_pred[i] == 1 else 'Negative'
    print('\nTweet:', ' '.join(X_test[i]))
    print('Actual:', actual, '| Predicted:', pred)

## 12. Predict sentiment of a new tweet

In [ ]:
new_tweet = 'I really love this project :)'
cleaned = clean_tweet(new_tweet)
tokens = tokenizer.tokenize(cleaned)
features = vectorizer.transform([tokens])
prediction = int(model.predict(features)[0])
print('Tweet:', new_tweet)
print('Sentiment:', 'Positive :)' if prediction == 1 else 'Negative :(')

## 13. Save model to a `.pkl` file

**Security:** Only open `.pkl` files from sources you trust. Cloudpickle is used because the vectorizer contains a lambda function.

In [ ]:
import cloudpickle

model_package = {
    'model': model,
    'vectorizer': vectorizer,
    'tokenizer': tokenizer,
    'clean_tweet': clean_tweet,
}
filename = 'twitter_sentiment_model.pkl'
with open(filename, 'wb') as f:
    cloudpickle.dump(model_package, f)
print('Model saved:', filename)

# In Google Colab only, download the file to your computer:
from google.colab import files
files.download(filename)

## 14. Load the saved model and predict again

In [ ]:
with open('twitter_sentiment_model.pkl', 'rb') as f:
    loaded_package = cloudpickle.load(f)

loaded_model = loaded_package['model']
loaded_vectorizer = loaded_package['vectorizer']
loaded_tokenizer = loaded_package['tokenizer']
loaded_clean_tweet = loaded_package['clean_tweet']

new_tweet = 'I am so happy today :)'
tokens = loaded_tokenizer.tokenize(loaded_clean_tweet(new_tweet))
pred = int(loaded_model.predict(loaded_vectorizer.transform([tokens]))[0])
print('Model loaded successfully!')
print('Tweet:', new_tweet)
print('Sentiment:', 'Positive :)' if pred == 1 else 'Negative :(')

## 15. Optional: launch a Gradio demo

⚠️ In Colab, `share=True` creates a **temporary public URL** that others may use. Only run this cell if you want that.

In [ ]:
import gradio as gr

def predict_sentiment(tweet):
    if not tweet or not tweet.strip():
        return 'Please enter a tweet.'
    tokens = loaded_tokenizer.tokenize(loaded_clean_tweet(tweet))
    features = loaded_vectorizer.transform([tokens])
    prediction = int(loaded_model.predict(features)[0])
    return 'Positive :)' if prediction == 1 else 'Negative :('

demo = gr.Interface(
    fn=predict_sentiment,
    inputs=gr.Textbox(lines=3, label='Your Tweet', placeholder='Enter a tweet...'),
    outputs=gr.Textbox(label='Predicted sentiment'),
    title='Twitter Sentiment Analysis',
    description='Binary sentiment analysis with TF-IDF and Naive Bayes',
)
demo.launch(share=True)

## Summary / Possible improvements

- Measure performance on text **without emoticons** to test robustness.
- Add a neutral class using an appropriately labeled dataset.
- Test against tweets from a different source to check generalization.
- Add optional hashtag/topic summaries with data you have permission to collect.
- Never assume positive/negative class labels are objective for sarcasm, mixed emotions, or context-dependent tweets.